# Extractive vs. Abstractive Summarization Analysis

In [6]:
import torch
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize
from tqdm import tqdm
from transformers import AutoTokenizer, BartForConditionalGeneration

nltk.download("punkt_tab")
nltk.download("punkt")

bart_id = "facebook/bart-large-cnn"

bart_tokenizer = AutoTokenizer.from_pretrained(bart_id)
bart_model = BartForConditionalGeneration.from_pretrained(bart_id)

source_text = """
The Apollo 11 mission was the first spaceflight that landed humans on the Moon. 
Commander Neil Armstrong and lunar module pilot Buzz Aldrin landed the Apollo Lunar Module Eagle 
on July 20, 1969. Armstrong became the first person to step onto the lunar surface six hours 
and 39 minutes later on July 21; Aldrin joined him 19 minutes later. They spent about 
two and a quarter hours together outside the spacecraft, and they collected 47.5 pounds 
of lunar material to bring back to Earth.
"""

[nltk_data] Downloading package punkt_tab to /Users/jose/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package punkt to /Users/jose/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
Loading weights: 100%|██████████| 511/511 [00:00<00:00, 1290.78it/s, Materializing param=model.encoder.layers.11.self_attn_layer_norm.weight]   


## Abstractive (standard BART)

In [15]:
def get_abstractive_summary(text, tokenizer, model):
    inputs = tokenizer([text], max_length=1024, return_tensors="pt", truncation=True)
    summary_ids = model.generate(inputs["input_ids"], num_beams=4, max_length=150, early_stopping=True)
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

## Extractive (sentence scoring)

In [16]:
def get_extractive_summary(source_text, tokenizer, model, top_k=3):
    sentences = sent_tokenize(source_text)
    sentence_scores = []

    inputs = tokenizer(source_text, max_length=1024, return_tensors="pt", truncation=True)

    for cand in sentences:
        labels = tokenizer(cand, return_tensors="pt").input_ids

        with torch.no_grad():
            outputs = model(input_ids=inputs["input_ids"], labels=labels)
            loss = outputs.loss.item()
            sentence_scores.append((loss, cand))

    sentence_scores.sort(key=lambda x: x[0])

    return " ".join([s[1] for s in sentence_scores[:top_k]])

## Generate Summaries

In [18]:
abstractive_result = get_abstractive_summary(source_text, bart_tokenizer, bart_model)
extractive_result = get_extractive_summary(source_text, bart_tokenizer, bart_model)

## Evaluation

In [19]:
from rouge_score import rouge_scorer

def calculate_novelty(source, summary):
    source_words = set(source.split())
    summary_words = set(summary.split())
    novel_words = [w for w in summary_words if w not in source_words]
    if len(summary_words) == 0:
        return 0.0
    return len(novel_words) / len(summary_words) * 100

scorer = rouge_scorer.RougeScorer(["rouge1","rougeL"], use_stemmer=True)

## Results

In [20]:
print("Source Text:\n", source_text)
# Should have a 10-30% novelty score and should have a lower rouge score because there will be less overlap with the source text
print("\nAbstractive Summary:\n", abstractive_result)
print("\nNovelty Score: {:.2f}%".format(calculate_novelty(source_text, abstractive_result)))
print("\nROUGE Scores:\n", scorer.score(source_text, abstractive_result))
# Should have a low novelty score since it pull sentences directly from the source and should have high rouge scores since there should be overlap
print("\nExtractive Summary:\n", extractive_result)
print("\nNovelty Score: {:.2f}%".format(calculate_novelty(source_text, extractive_result)))
print("\nROUGE Scores:\n", scorer.score(source_text, extractive_result))

Source Text:
 
The Apollo 11 mission was the first spaceflight that landed humans on the Moon. 
Commander Neil Armstrong and lunar module pilot Buzz Aldrin landed the Apollo Lunar Module Eagle 
on July 20, 1969. Armstrong became the first person to step onto the lunar surface six hours 
and 39 minutes later on July 21; Aldrin joined him 19 minutes later. They spent about 
two and a quarter hours together outside the spacecraft, and they collected 47.5 pounds 
of lunar material to bring back to Earth.


Abstractive Summary:
 The Apollo 11 mission was the first spaceflight that landed humans on the Moon. Commander Neil Armstrong and lunar module pilot Buzz Aldrin landed the Apollo Lunar Module Eagle on July 20, 1969. Armstrong became the first person to step onto the lunar surface six hours and 39 minutes later on July 21.

Novelty Score: 2.50%

ROUGE Scores:
 {'rouge1': Score(precision=1.0, recall=0.6235294117647059, fmeasure=0.7681159420289855), 'rougeL': Score(precision=1.0, recall=0.